# 2025 DL Lab8: RL Assignment_Super Mario World

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 314831000.

**Your Answer:**    
Hi I'm 宋尚勳, 314834014.

## Overview
This project implements a **Deep Reinforcement Learning** pipeline to train an autonomous agent for Super Mario World. Leveraging the **Proximal Policy Optimization (PPO)** algorithm, the system interacts with the **stable-retro** environment to master the YoshiIsland1 level. Key components include a custom Vision Backbone for extracting features from raw pixel data and a suite of Environment Wrappers that handle frame preprocessing, action discretization, and reward shaping to facilitate efficient learning.

## Imports

In [1]:
import os
import retro
import numpy as np
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

from eval import evaluate_policy, record_video
from custom_policy import VisionBackbonePolicy, CustomPPO

## Configuration

In [2]:
# Game Settings
GAME = "SuperMarioWorld-Snes"
STATE = "YoshiIsland1"

# Training Settings
TOTAL_STEPS = 1_000_000
TRAIN_CHUNK = 50_000
N_ENVS = 2
LEARNING_RATE = 1e-4

# Evaluation & Recording Settings
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 18000
RECORD_STEPS = 18000

# Directories
LOG_DIR = "./runs_smw"
VIDEO_DIR = os.path.join(LOG_DIR, "videos")
CKPT_DIR = os.path.join(LOG_DIR, "checkpoints")
TENSORBOARD_LOG = os.path.join(LOG_DIR, "tb")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

## Environment Functions

In [3]:
from wrappers import make_base_env
def _make_env_thunk(game: str, state: str):
    """Return a function that creates an environment (for multiprocessing)."""
    def _thunk():
        return make_base_env(game, state)
    return _thunk

def make_vec_env(game: str, state: str, n_envs: int, use_subproc: bool = True):
    """Create a vectorized environment (multiple envs running in parallel)."""
    env_fns = [_make_env_thunk(game, state) for _ in range(n_envs)]
    
    if use_subproc and n_envs > 1:
        vec_env = SubprocVecEnv(env_fns)
    else:
        vec_env = DummyVecEnv(env_fns)

    return vec_env


## Initialize Env & Model

In [4]:
# 1. Create Training Environment
train_env = make_vec_env(GAME, STATE, n_envs=N_ENVS)
print(f"Environment created: {GAME} - {STATE} with {N_ENVS} parallel envs.")

# 2. Initialize Model
model = CustomPPO(
    VisionBackbonePolicy,
    train_env,
    policy_kwargs=dict(normalize_images=False),
    n_epochs=10,
    n_steps=512,
    batch_size=512,
    learning_rate=LEARNING_RATE,
    verbose=1,
    gamma=0.99,
    kl_coef=1,
    clip_range=0.5,
    tensorboard_log=TENSORBOARD_LOG,
)

Environment created: SuperMarioWorld-Snes - YoshiIsland1 with 2 parallel envs.
Using cuda:0 device


## Training Loop

In [5]:
best_mean = -1e18
trained = 0
round_idx = 0

try:
    while trained < TOTAL_STEPS:
        round_idx += 1
        chunk = min(TRAIN_CHUNK, TOTAL_STEPS - trained)

        print(f"\n=== Round {round_idx} | Learn {chunk} steps (Total trained: {trained}) ===")
        
        # --- Train ---
        model.learn(total_timesteps=chunk, reset_num_timesteps=False)
        trained += chunk

        # --- Save Checkpoint ---
        ckpt_path = os.path.join(CKPT_DIR, f"CustomPPO_step_{trained}.zip")
        model.save(ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

        # --- Evaluate ---
        mean_ret, best_ret = evaluate_policy(
            model,
            GAME,
            STATE,
            n_episodes=EVAL_EPISODES,
            max_steps=EVAL_MAX_STEPS,
        )
        print(f"[EVAL] Mean Return: {mean_ret:.3f}, Best Return: {best_ret:.3f}")

        # --- Save Best Model ---
        if mean_ret > best_mean:
            best_mean = mean_ret
            best_path = os.path.join(LOG_DIR, "best_model.zip")
            model.save(best_path)
            print(f"New best record. Saved to {best_path}")

        # --- Record Video ---
        record_video(
            model,
            GAME,
            STATE,
            VIDEO_DIR,
            video_len=RECORD_STEPS,
            prefix=f"step_{trained}_mean_{mean_ret:.2f}",
        )

except KeyboardInterrupt:
    print("\nTraining interrupted manually.")

finally:
    train_env.close()
    print("Training finished. Environment closed.")


=== Round 1 | Learn 50000 steps (Total trained: 0) ===
Logging to ./runs_smw/tb/MyPPO_0


Output()

-----------------------------
| time/              |      |
|    fps             | 222  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 1024 |
-----------------------------
---------------------------------------
| time/                   |           |
|    fps                  | 123       |
|    iterations           | 2         |
|    time_elapsed         | 16        |
|    total_timesteps      | 2048      |
| train/                  |           |
|    approx_kl            | -0.000137 |
|    clip_fraction        | 0         |
|    clip_range           | 0.5       |
|    entropy_loss         | -2.48     |
|    explained_variance   | 0.000669  |
|    learning_rate        | 0.0001    |
|    loss                 | 0.743     |
|    n_updates            | 10        |
|    policy_gradient_loss | -0.00224  |
|    value_loss           | 2.19      |
---------------------------------------
--------------------------------------
| time/                   |    

Saved checkpoint: ./runs_smw/checkpoints/CustomPPO_step_50000.zip
[EVAL] Mean Return: 16.000, Best Return: 16.000
New best record. Saved to ./runs_smw/best_model.zip
Saved video to ./runs_smw/videos/step_50000_mean_16.00.mp4

=== Round 2 | Learn 50000 steps (Total trained: 50000) ===
Logging to ./runs_smw/tb/MyPPO_0


Output()

...

-------------------------------
| time/              |        |
|    fps             | 245    |
|    iterations      | 1      |
|    time_elapsed    | 4      |
|    total_timesteps | 904192 |
-------------------------------
--------------------------------------
| time/                   |          |
|    fps                  | 129      |
|    iterations           | 2        |
|    time_elapsed         | 15       |
|    total_timesteps      | 905216   |
| train/                  |          |
|    approx_kl            | 0.0119   |
|    clip_fraction        | 0.00566  |
|    clip_range           | 0.5      |
|    entropy_loss         | -1.38    |
|    explained_variance   | 0.981    |
|    learning_rate        | 0.0001   |
|    loss                 | 4.59     |
|    n_updates            | 8830     |
|    policy_gradient_loss | -0.00302 |
|    value_loss           | 17.1     |
--------------------------------------
--------------------------------------
| time/                   |        

Saved checkpoint: ./runs_smw/checkpoints/CustomPPO_step_950000.zip
[EVAL] Mean Return: 16.000, Best Return: 16.000
Saved video to ./runs_smw/videos/step_950000_mean_16.00.mp4

=== Round 20 | Learn 50000 steps (Total trained: 950000) ===
Logging to ./runs_smw/tb/MyPPO_0


Output()

-------------------------------
| time/              |        |
|    fps             | 244    |
|    iterations      | 1      |
|    time_elapsed    | 4      |
|    total_timesteps | 954368 |
-------------------------------
--------------------------------------
| time/                   |          |
|    fps                  | 129      |
|    iterations           | 2        |
|    time_elapsed         | 15       |
|    total_timesteps      | 955392   |
| train/                  |          |
|    approx_kl            | 0.236    |
|    clip_fraction        | 0.0742   |
|    clip_range           | 0.5      |
|    entropy_loss         | -1.2     |
|    explained_variance   | 0.602    |
|    learning_rate        | 0.0001   |
|    loss                 | 60.6     |
|    n_updates            | 9320     |
|    policy_gradient_loss | -0.038   |
|    value_loss           | 193      |
--------------------------------------
--------------------------------------
| time/                   |        

Saved checkpoint: ./runs_smw/checkpoints/CustomPPO_step_1000000.zip
[EVAL] Mean Return: 16.000, Best Return: 16.000
Saved video to ./runs_smw/videos/step_1000000_mean_16.00.mp4
Training finished. Environment closed.


## Display Video

In [6]:
from IPython.display import Video
import glob

list_of_files = glob.glob(os.path.join(VIDEO_DIR, '*.mp4')) 
if list_of_files:
    latest_file = max(list_of_files, key=os.path.getctime)
    print(f"Playing: {latest_file}")
    display(Video(latest_file, embed=True, width=600))
else:
    print("No videos found yet.")

Playing: ./runs_smw/videos/step_1000000_mean_16.00.mp4
